# Entrenamiento de Modelados

In [47]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Importación de los datos

In [48]:
ruta = Path("..") / "data" / "cleaned"/ "train_clean.csv"

train = pd.read_csv(ruta)

In [49]:
train_x = train.drop(columns="Class")
train_y = train["Class"]

## Estrategia de validación

In [ ]:
def metricas_umbral(y, prob):
    pred = (prob >= UMBRAL).astype(int)
    return {
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
    }

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "auroc": "roc_auc",
    "auprc": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

def evaluar(modelo, param):
    res_cv = cross_validate(
        param.best_estimator_, train_x, train_y, cv=cv,
        return_estimator=True, return_indices=True, n_jobs=-1)

    proba = np.zeros(len(train_y))
    for est, idx in zip(res_cv["estimator"], res_cv["indices"]["test"]):
        proba[idx] = est.predict_proba(train_x.iloc[idx])[:, 1]

    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 5))
    RocCurveDisplay.from_cv_results(res_cv, train_x, train_y, name=modelo, ax=ax_roc)
    PrecisionRecallDisplay.from_cv_results(res_cv, train_x, train_y, name=modelo, ax=ax_pr)
    ax_roc.set_title("Curva ROC")
    ax_pr.set_title("Curva Precision-Recall")
    fig.suptitle(modelo)
    fig.tight_layout()
    plt.show()

    i = param.best_index_
    res = param.cv_results_

    fila = {
        "modelo": modelo,
        "auprc": res["mean_test_auprc"][i],
        "auprc_std": res["std_test_auprc"][i],
        "auroc": res["mean_test_auroc"][i],
        "auroc_std": res["std_test_auroc"][i],
        "umbral": UMBRAL,
        **metricas_umbral(train_y, proba),
        "auprc_train": res["mean_train_auprc"][i],
        "brecha": res["mean_train_auprc"][i] - res["mean_test_auprc"][i],
        "tiempo_fit_s": res["mean_fit_time"][i],
        "mejores_params": param.best_params_
    }

    return proba, fila

In [52]:
def matriz_confusion(modelo, y, prob):
    pred = (prob >= UMBRAL).astype(int)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y, pred, ax=ax, colorbar=False, values_format="d")
    ax.set_title(f"{modelo} - Matriz de confusión (umbral = {UMBRAL})")

    fig.tight_layout()
    plt.show()

## Busqueda de Hiperparametros

In [53]:
def buscar_param(modelo, params):
    busqueda = RandomizedSearchCV(
        modelo, param_distributions=params, n_iter=N_ITER, cv=cv,
        scoring={"auroc": "roc_auc", "auprc": "average_precision"},
        refit="auprc", return_train_score=True,
        random_state=42, n_jobs=-1)
    busqueda.fit(train_x, train_y)
    return busqueda

## Modelado

In [54]:
UMBRAL = 0.5
N_ITER = 5 

In [55]:
modelos = {
    "Dummy": (DummyClassifier(strategy="prior"), {}),
    "Regresión Logística": (
        LogisticRegression(max_iter=1000),
        {"C": [0.01, 0.1, 1], "class_weight": [None, "balanced"]}),
    "RandomForest": (
        RandomForestClassifier(max_samples=0.3, random_state=42, n_jobs=1),
        {"max_depth": [5, 10, None], "n_estimators": [50, 100]}),
    "XGBoost": (
        XGBClassifier(random_state=42, n_jobs=1),
        {"max_depth": [3, 5], "learning_rate": [0.05, 0.1]}),
    "LightGBM": (
        LGBMClassifier(random_state=42, n_jobs=1, verbose=-1),
        {"num_leaves": [15, 31], "learning_rate": [0.05, 0.1]}),
    "MLP": (
        MLPClassifier(max_iter=200, early_stopping=True, random_state=42),
        {"hidden_layer_sizes": [(32,), (64, 32)], "alpha": [0.0001, 0.01]}),
}